In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize


In [ ]:
# 전처리 및 특징 추출 파라미터
sampling_rate = 32000  # 샘플링 속도 (32kHz)
fft_size = 1728  # FFT 사이즈
window_length = 108  # 해밍 윈도우 길이 (108ms)
hop_length = 10  # 윈도우 시프트 (10ms)

In [ ]:
def preprocess_audio(audio_file):
    # 1. Load audio file
    y, sr = librosa.load(audio_file, sr=sampling_rate)

    # 2. Extract spectrogram
    spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=fft_size, hop_length=hop_length, window='hamming')
    spectrogram_db = librosa.amplitude_to_db(spectrogram, ref=np.max)  # 로그 스케일 변환
    
    mean_spectrum = np.mean(spectrogram_db, axis = 1)

    # 4. Z-normalization
    spectrogram = normalize(spectrogram, norm='l2', axis=1)

    return spectrogram

In [ ]:
# 오디오 샘플이 있는 디렉토리 경로 설정
audio_directory = "../data/unlabeled_audio"
audio_files = [os.path.join(audio_directory, f) for f in os.listdir(audio_directory) if f.endswith('.ogg')]

In [ ]:
# 스펙트로그램을 저장할 리스트 초기화
spectrograms = []

# 모든 오디오 파일에 대해 스펙트로그램 생성
for audio_file in audio_files:
    spectrogram = preprocess_audio(audio_file)
    spectrograms.append(spectrogram)

In [ ]:
import random

# 무작위로 선택할 파일의 개수 설정
num_samples = 10
random_files = random.sample(spectrograms, num_samples)

# 스펙트로그램 시각화
fig, axes = plt.subplots(nrows=num_samples, figsize=(10, 4 * num_samples))
for i, spec in enumerate(random_files):
    librosa.display.specshow(spec, sr=sampling_rate, hop_length=hop_length, x_axis='time', y_axis='mel', ax=axes[i],cmap='magma')
    axes[i].set_title(f'Spectrogram {i+1}')
    axes[i].set_ylabel('Mel Frequency')
    axes[i].set_xlabel('Time')
    fig.colorbar(axes[i].collections[0], ax=axes[i], format='%+2.0f dB')

plt.tight_layout()
plt.show()